# Pipeline de dados - Imoveis em BH 

## Objetivo : Extração ,tratamento e analise

### Eduardo Cosme

**Observação: Como dentro do databricks não me permite instalar o pacote Python  cloudscraper este projeto sera feito de 02 etapas a extração sera via Jupyter e tratamento camadas bronze/prata/ouro sera feita no Databricks

**Site utilizado no projeto :https://www.zapimoveis.com.br/


In [ ]:
##importar os pacotes 

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import os
import requests
from bs4 import BeautifulSoup


In [27]:
## Usar a API extraida do site 28/03/2026
url = 'https://glue-api.zapimoveis.com.br/v2/listings'
response = requests.get(url)
html_content = response.content
print(html_content)




b'<!DOCTYPE html>\n<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->\n<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->\n<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->\n<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->\n<head>\n<title>Attention Required! | Cloudflare</title>\n<meta charset="UTF-8" />\n<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />\n<meta http-equiv="X-UA-Compatible" content="IE=Edge" />\n<meta name="robots" content="noindex, nofollow" />\n<meta name="viewport" content="width=device-width,initial-scale=1" />\n<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/cf.errors.css" />\n<!--[if lt IE 9]><link rel="stylesheet" id=\'cf_styles-ie-css\' href="/cdn-cgi/styles/cf.errors.ie.css" /><![endif]-->\n<style>body{margin:0;padding:0}</style>\n\n\n<!--[if gte IE 10]><!-->\n<script>\n  if (!navigator.cookieEnabled) {\n    window.addEventListen

In [30]:
## Usar a pacote BeautifulSoup , para tornar a pagina legivel e fazer buscas 
site = BeautifulSoup(html_content,'html.parser')
print(site)

<!DOCTYPE html>

<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->
<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->
<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->
<head>
<title>Attention Required! | Cloudflare</title>
<meta charset="utf-8"/>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<meta content="IE=Edge" http-equiv="X-UA-Compatible"/>
<meta content="noindex, nofollow" name="robots"/>
<meta content="width=device-width,initial-scale=1" name="viewport"/>
<link href="/cdn-cgi/styles/cf.errors.css" id="cf_styles-css" rel="stylesheet"/>
<!--[if lt IE 9]><link rel="stylesheet" id='cf_styles-ie-css' href="/cdn-cgi/styles/cf.errors.ie.css" /><![endif]-->
<style>body{margin:0;padding:0}</style>
<!--[if gte IE 10]><!-->
<script>
  if (!navigator.cookieEnabled) {
    window.addEventListener('DOMContentLoaded', function

***Ponto de aprendizado***

Localizei a API correta onde fica os dados dos imoveis, porem o site tem uma proteção para acessos automatizados o que é correto ja que não é uma API publica para contornar este problema vou utilizar um pacote cloudscraper. 

Como para este projeto não estou usando dados ja estruturados e o objetivo é buscar na web para criar uma pipeline de dados para ser tratado em camadas . 

In [53]:
## importar o pacote 
import cloudscraper
import json





In [56]:
# Criar scraper com configurações personalizadas

url = "https://glue-api.zapimoveis.com.br/v2/listings"

params = {
    'business': 'RENTAL',
    'addressCity': 'Belo Horizonte',
    'addressState': 'Minas Gerais',
    'addressLocationId': 'BR>Minas Gerais>NULL>Belo Horizonte',
    'addressType': 'city',
    'listingType': 'USED',
    'page': 1,
    'size': 1,
    'categoryPage': 'RESULT'
}


scraper = cloudscraper.create_scraper(
    browser={
        'browser': 'chrome',
        'platform': 'windows',
        'mobile': False
    }
)

# Headers personalizados

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.3',
    'Accept-Language': 'pt-BR,pt;q=0.9,en;q=0.8',
    'Accept-Encoding': 'gzip, deflate, br',
    'x-domain': '.zapimoveis.com.br'
}

# Fazer requisição com headers
try:
    site = scraper.get(url,
        headers=headers,
        params=params,               
        timeout=30
    )
    
    if site.status_code == 200:
        print("Requisição bem sucedida!")
        print(site.json())
        
    else:
         print(f'Status:{site.status_code}')
         print(site.text)
        
except Exception as e:
    print(f"Erro:{e}")

Status:403
<!DOCTYPE html>
<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->
<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->
<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->
<head>
<title>Attention Required! | Cloudflare</title>
<meta charset="UTF-8" />
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<meta http-equiv="X-UA-Compatible" content="IE=Edge" />
<meta name="robots" content="noindex, nofollow" />
<meta name="viewport" content="width=device-width,initial-scale=1" />
<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/cf.errors.css" />
<!--[if lt IE 9]><link rel="stylesheet" id='cf_styles-ie-css' href="/cdn-cgi/styles/cf.errors.ie.css" /><![endif]-->
<style>body{margin:0;padding:0}</style>


<!--[if gte IE 10]><!-->
<script>
  if (!navigator.cookieEnabled) {
    window.addEventListener('DOMConten

In [57]:
python version

SyntaxError: invalid syntax (1241365062.py, line 1)